# 15. LCA ライフサイクルアセスメント（Life Cycle Assessment） — 練習問題

**対象技術**: 量子コンピューティング

ライフサイクルアセスメント（LCA）は、技術のライフサイクル全段階（製造・運用・廃棄）の資源消費と環境排出を積み上げ、特性化係数で地球温暖化係数（GWP, CO2当量）に換算して環境インパクトを定量評価する手法である。このノートブックでは量子コンピュータと古典スーパーコンピュータを共通の機能単位「量子化学計算1回の実行」の下で比較する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## インベントリと特性化係数の定義

量子計算機と古典スパコンの段階別インベントリ（機能単位＝量子化学計算1回あたり、kWh換算）と、電力1kWhあたりのCO2排出量（特性化係数）を定義する。

In [ ]:
STAGES = ["製造", "運用", "廃棄"]

# 量子コンピュータのインベントリ(S=1 基準ケース)。単位は kWh / ジョブ。
QC_INVENTORY_BASE = {
    "製造": 120.0,   # 希釈冷凍機・超伝導チップ等の製造を配分した値
    "運用": 200.0,   # 連続冷却電力(S=1基準。うち待機冷却分は QC_IDLE)
    "廃棄": 15.0,    # 極低温機器の解体・素材回収
}
# 運用エネルギーのうち、計算の有無に関係なく常時かかる待機冷却分。
QC_IDLE = 90.0       # kWh / ジョブ

# 古典スーパーコンピュータのインベントリ。
CLASSIC_INVENTORY = {
    "製造": 60.0,
    "運用": 260.0,
    "廃棄": 8.0,
}

# 特性化係数: 電力1kWhあたりのCO2排出量(電源構成依存の想定値)。
GWP_FACTOR = 0.45    # kg-CO2e / kWh

print(f"機能単位: 「ある量子化学計算を1回実行する」")
print(f"特性化係数: {GWP_FACTOR} kg-CO2e/kWh (電源構成依存の想定値)")

## 段階別 GWP の算出

量子計算機の運用エネルギーは、待機冷却分（固定）＋計算分／S で表す。優位倍率 S が大きいほど計算時間が短縮され計算分が減る。各段階のエネルギーに特性化係数を掛けて GWP を算出する関数を定義する。

In [ ]:
def qc_operational_energy(speedup):
    """量子計算機の運用エネルギーを優位倍率 S に応じて算出する。
    運用エネルギー = 待機冷却分(固定) + 計算分 / S。
    """
    compute_part = QC_INVENTORY_BASE["運用"] - QC_IDLE
    return QC_IDLE + compute_part / speedup


def total_gwp_qc(speedup):
    """量子計算機の段階別および総GWP(kg-CO2e)を優位倍率 S で算出する。"""
    manufacture = QC_INVENTORY_BASE["製造"] * GWP_FACTOR
    operation = qc_operational_energy(speedup) * GWP_FACTOR
    disposal = QC_INVENTORY_BASE["廃棄"] * GWP_FACTOR
    return manufacture, operation, disposal, manufacture + operation + disposal


def total_gwp_classic():
    """古典スパコンの段階別および総GWP(kg-CO2e)を算出する。"""
    manufacture = CLASSIC_INVENTORY["製造"] * GWP_FACTOR
    operation = CLASSIC_INVENTORY["運用"] * GWP_FACTOR
    disposal = CLASSIC_INVENTORY["廃棄"] * GWP_FACTOR
    return manufacture, operation, disposal, manufacture + operation + disposal


qc_m, qc_o, qc_d, qc_total = total_gwp_qc(1.0)
cl_m, cl_o, cl_d, cl_total = total_gwp_classic()

print("[段階別 GWP] (kg-CO2e / ジョブ, 量子は優位倍率 S=1 の基準ケース)")
print("-" * 70)
print(f"  {'段階':<8}{'量子計算機':>16}{'古典スパコン':>16}")
print(f"  {'製造':<8}{qc_m:>16.2f}{cl_m:>16.2f}")
print(f"  {'運用':<8}{qc_o:>16.2f}{cl_o:>16.2f}")
print(f"  {'廃棄':<8}{qc_d:>16.2f}{cl_d:>16.2f}")
print("  " + "-" * 44)
print(f"  {'総計':<8}{qc_total:>16.2f}{cl_total:>16.2f}")

qc_stages = {"製造": qc_m, "運用": qc_o, "廃棄": qc_d}
hotspot = max(qc_stages, key=qc_stages.get)
print(f"\n[ホットスポット] 量子計算機の最大負荷段階 = 『{hotspot}』")
print(f"  運用段階の総排出 {qc_o:.1f} のうち待機冷却分 "
      f"{QC_IDLE * GWP_FACTOR:.1f} は計算の有無に関わらず常時発生する。")

## ブレークイーブン分析

総GWPが古典スパコンと一致する優位倍率 S* を二分探索で求める。QC の総GWPは S の単調減少関数なので二分探索が使える。

In [ ]:
def find_breakeven(s_min=0.5, s_max=50.0):
    """総GWPが古典スパコンと一致する優位倍率 S* を二分探索で求める。"""
    classic_total = total_gwp_classic()[3]
    lo, hi = s_min, s_max
    if total_gwp_qc(s_min)[3] < classic_total:
        return None  # S_min でも既に量子が有利
    if total_gwp_qc(s_max)[3] > classic_total:
        return None  # S_max でも量子が不利のまま
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if total_gwp_qc(mid)[3] > classic_total:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


breakeven = find_breakeven()
print("[ブレークイーブン分析] 量子側の計算速度優位倍率 S を変数に")
print("-" * 70)
for s in [1, 2, 5, 10, 20]:
    qt = total_gwp_qc(s)[3]
    verdict = "量子が有利" if qt < cl_total else "量子が不利"
    print(f"  S={s:>3}: 量子総GWP={qt:>8.2f}  (古典={cl_total:.2f}) -> {verdict}")
print("-" * 70)
if breakeven is None:
    print("  探索範囲内でブレークイーブン点なし(常に一方が有利)。")
else:
    print(f"  ブレークイーブン点 S* = {breakeven:.2f}")
    print(f"  -> S > {breakeven:.2f} なら量子計算機が環境的に有利、")
    print(f"     S < {breakeven:.2f} なら製造負荷と常時冷却の不利が効いて")
    print("     量子計算機の総排出はかえって古典スパコンを上回る。")

## 可視化1: ライフサイクル段階別 GWP の積み上げ棒グラフ

量子計算機（S=1 基準ケース）と古典スパコンの段階別 GWP を、製造・運用・廃棄の積み上げ棒グラフで並置する。どの段階が支配的か（ホットスポット）が一目で分かる。

In [ ]:
stages_en = ["Manufacture", "Operation", "Disposal"]
qc_vals = [qc_m, qc_o, qc_d]
cl_vals = [cl_m, cl_o, cl_d]
colors = ["#7fb8d6", "#3f7ca6", "#1f3f5c"]

fig, ax = plt.subplots(figsize=(6, 5))
x = [0, 1]
labels = ["Quantum (S=1)", "Classical"]
bottom_qc = 0.0
bottom_cl = 0.0
for k in range(3):
    ax.bar(x[0], qc_vals[k], 0.55, bottom=bottom_qc, color=colors[k],
           label=stages_en[k])
    ax.bar(x[1], cl_vals[k], 0.55, bottom=bottom_cl, color=colors[k])
    bottom_qc += qc_vals[k]
    bottom_cl += cl_vals[k]
ax.text(x[0], bottom_qc + 1, f"{bottom_qc:.1f}", ha="center", fontsize=9)
ax.text(x[1], bottom_cl + 1, f"{bottom_cl:.1f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("GWP per job (kg-CO2e)")
ax.set_title("LCA: life-cycle stage GWP (Quantum vs Classical)")
ax.legend()
fig.tight_layout()
plt.show()

## 可視化2: 優位倍率に対する総排出曲線とブレークイーブン点

優位倍率 S に対する量子計算機の総GWP曲線を描き、古典スパコンの総GWP（一定）と交わるブレークイーブン点 S* を注記する。

In [ ]:
s_values = np.linspace(0.5, 30.0, 200)
qc_curve = np.array([total_gwp_qc(s)[3] for s in s_values])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(s_values, qc_curve, label="Quantum computer (total GWP)",
        color="#1f3f5c", linewidth=2)
ax.axhline(cl_total, color="#c0504d", linestyle="--",
           label="Classical supercomputer (total GWP)")
if breakeven is not None:
    ax.axvline(breakeven, color="#7f7f7f", linestyle=":")
    ax.plot([breakeven], [cl_total], "o", color="#c0504d", zorder=5)
    ax.text(breakeven + 0.5, cl_total * 1.07,
            f"break-even S* = {breakeven:.2f}", fontsize=9)
ax.set_xlabel("Quantum speedup factor S")
ax.set_ylabel("Total GWP per job (kg-CO2e)")
ax.set_title("LCA: total emissions vs quantum speedup")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

未来デザイン論文においてライフサイクルアセスメント（LCA）は、ある技術の環境負荷を、原料採取から廃棄に至る全段階にわたって定量的に積み上げ、代替技術と比較するために用いられる。論文はまず機能単位を定め、システム境界を画定し、段階別のインベントリに特性化係数を掛けて環境影響を集計する。そこで生まれる結論は、ライフサイクル境界内の定量比較という明快な型をとる。「この技術の環境負荷は代替技術よりXだけ低い（あるいは高い）」「ある性能優位の倍率を超えると総排出が逆転する」というブレークイーブンの言明が、数値の裏づけとともに提示される。

この手法が結論に持ち込む規定力は、何よりも境界設定にある。機能単位の定義とシステム境界の引き方が、比較の土俵そのものを決め、ひいては結論の符号すら左右する。境界の内側に入れた段階の負荷だけが集計され、境界の外に置かれた影響——とりわけ社会的・経済的な帰結や、雇用や分配への作用——は構造的に不可視となる。LCA は環境という一領域に焦点を絞ることで定量的な厳密さを獲得するが、その厳密さは射程の狭さと引き換えに得られている。

時間観の面では、LCA は将来の運用条件——電力構成の脱炭素化の進み方など——を前提値として固定する。未来を選択や設計の対象としてではなく、係数として与えられたものとして扱うため、結論はその前提の妥当性に強く依存する。価値の所在は、特性化係数とどの環境影響カテゴリを採るかという選択に埋め込まれる。地球温暖化係数だけを見るか、資源枯渇や生態系影響まで含めるかで、同じ技術の評価は変わりうる。したがってこの手法を用いた論文は、数値の客観性という外観を持ちながら、境界・前提・係数の選択に判断が凝縮されており、その透明な開示とパラメータ感度分析を欠けば、結論は限定された土俵での部分最適を全体の優劣であるかのように語ってしまう。

## 発展課題

**課題A**: 製造段階のインベントリや特性化係数、待機冷却電力などのパラメータを変化させてブレークイーブン点 S* がどう動くか感度分析せよ。結論（量子が有利になる条件）が前提にどれだけ依存するかを論ぜよ。

**課題B**: ヘリウム3 など希少資源の枯渇制約をモデルに追加せよ。資源消費に上限や逼迫係数を設け、資源制約が GWP 比較や S* にどう影響するかを評価せよ。